Overview & Teori DasarPada bab ini, dipelajari cara mengolah data teks (Natural Language Processing / NLP) agar dapat dipahami oleh neural network:   Natural Language Processing (NLP): Mengubah kata-kata menjadi angka agar komputer dapat memproses hubungan antar-kata secara intuitif.   Word Embeddings (Embedding Layer): Memetakan kata-kata dari format one-hot ke dalam ruang vektor berdimensi rendah (dense vector).   Word Correlation & Analogy: Kata-kata yang memiliki konteks serupa akan berada berdekatan dalam ruang vektor, memungkinkan kalkulasi matematis seperti:
   $$\text{king} - \text{man} + \text{woman} \approx \text{queen}$$

1. Dasar Embedding Layer dan Lookup Table

In [1]:
import numpy as np

# Kosakata sederhana (Vocabulary)
word2index = {'king': 0, 'queen': 1, 'man': 2, 'woman': 3}
index2word = {0: 'king', 1: 'queen', 2: 'man', 3: 'woman'}

# Inisialisasi Matriks Embedding (4 kata x 3 dimensi fitur)
np.random.seed(1)
embed_matrix = 2 * np.random.random((len(word2index), 3)) - 1

# Mengambil vektor embedding untuk kata 'king' (indeks 0) menggunakan Lookup
king_index = word2index['king']
king_embedding = embed_matrix[king_index]

print("Matriks Embeddings:\n", embed_matrix)
print("Vektor Embedding 'king':", king_embedding)

Matriks Embeddings:
 [[-0.16595599  0.44064899 -0.99977125]
 [-0.39533485 -0.70648822 -0.81532281]
 [-0.62747958 -0.30887855 -0.20646505]
 [ 0.07763347 -0.16161097  0.370439  ]]
Vektor Embedding 'king': [-0.16595599  0.44064899 -0.99977125]


Penjelasan Teoretis:Lookup Table vs One-Hot Matrix Multiplication: Mengalikan vektor one-hot berukuran besar dengan matriks bobot hanya akan menghasilkan baris matriks di indeks tersebut. Menggunakan operasi lookup (embed_matrix[index]) jauh lebih cepat dan hemat memori.   

2. Pelatihan Word Embeddings (Metode Skip-Gram Sederhana)

In [2]:
import numpy as np

def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

# Matriks Bobot Input->Hidden (Embeddings) dan Hidden->Output
dim_size = 3
weights_input = 2 * np.random.random((len(word2index), dim_size)) - 1
weights_output = 2 * np.random.random((dim_size, len(word2index))) - 1

alpha = 0.05
target_word = 'king'
context_word = 'man'

target_idx = word2index[target_word]
context_idx = word2index[context_word]

for iteration in range(50):
    # 1. FORWARD PASS
    hidden = weights_input[target_idx]              # Lookup embedding
    scores = hidden.dot(weights_output)             # Skor logits seluruh kata
    probs = softmax(scores)                         # Distribusi probabilitas

    # 2. COMPARE
    target_one_hot = np.zeros(len(word2index))
    target_one_hot[context_idx] = 1

    output_delta = probs - target_one_hot
    hidden_delta = output_delta.dot(weights_output.T)

    # 3. BACKPROPAGATION & UPDATE BOBOT
    weights_output -= alpha * np.outer(hidden, output_delta)
    weights_input[target_idx] -= alpha * hidden_delta

    if iteration % 10 == 9:
        loss = -np.log(probs[context_idx])
        print(f"Iterasi {iteration+1:2d} | Loss: {loss:.6f} | Probabilitas '{context_word}': {probs[context_idx]:.4f}")

Iterasi 10 | Loss: 0.975544 | Probabilitas 'man': 0.3770
Iterasi 20 | Loss: 0.494164 | Probabilitas 'man': 0.6101
Iterasi 30 | Loss: 0.258858 | Probabilitas 'man': 0.7719
Iterasi 40 | Loss: 0.151878 | Probabilitas 'man': 0.8591
Iterasi 50 | Loss: 0.099800 | Probabilitas 'man': 0.9050


Penjelasan Teoretis:

Predicting Context: Model dilatih untuk memprediksi kata-kata yang sering muncul di sekitar kata target.

Backpropagation pada Embedding: Hanya baris matriks embedding dari kata target (weights_input[target_idx]) yang diperbarui pada setiap langkah pelatihan.